# Fixed Egr1 CC Peak Overlap and Heatmap Pipeline

This notebook builds a merged union of male and female Egr1 WT peaks, expands peak windows slightly to capture near-miss overlaps, and then intersects with sex-biased male vs female and female vs male peaks. Finally, it generates a single `computeMatrix` output and heatmap for Male and Female Egr1 CC signal over the shared/male-biased/female-biased peak sets.

In [ ]:
import os
import subprocess
import pandas as pd

os.chdir('/scratch/rmlab/rmlab_shared3/l.llaci/Egr1_paper/testing_CC_forPaper')
print(f"Working directory: {os.getcwd()}")


In [ ]:
def read_bed(path):
    df = pd.read_csv(path, sep='\t', header=None, usecols=[0,1,2], names=['chr','start','end'])
    df["start"] = df["start"].astype(int)
    df["end"] = df["end"].astype(int)
    return df

def slop_bed(df, b):
    df = df.copy()
    df["start"] = (df["start"] - b).clip(lower=0)
    df["end"] = df["end"] + b
    return df

def merge_bed(df):
    df = df.sort_values(["chr", "start"]).reset_index(drop=True)
    merged = []
    for _, row in df.iterrows():
        if not merged or row["chr"] != merged[-1]["chr"] or row["start"] > merged[-1]["end"]:
            merged.append({"chr": row["chr"], "start": row["start"], "end": row["end"]})
        else:
            merged[-1]["end"] = max(merged[-1]["end"], row["end"])
    return pd.DataFrame(merged, columns=["chr", "start", "end"])

def overlapping_regions(df1, df2):
    df1 = df1.sort_values(["chr", "start"]).reset_index(drop=True)
    df2 = df2.sort_values(["chr", "start"]).reset_index(drop=True)
    rows = []
    j = 0
    for _, row1 in df1.iterrows():
        chr1, s1, e1 = row1["chr"], row1["start"], row1["end"]
        while j < len(df2) and (df2.loc[j, "chr"] < chr1 or (df2.loc[j, "chr"] == chr1 and df2.loc[j, "end"] <= s1)):
            j += 1
        k = j
        overlap_found = False
        while k < len(df2) and df2.loc[k, "chr"] == chr1 and df2.loc[k, "start"] < e1:
            if df2.loc[k, "end"] > s1:
                overlap_found = True
                break
            k += 1
        if overlap_found:
            rows.append(row1.to_dict())
    return pd.DataFrame(rows, columns=["chr", "start", "end"])

slop_bp = 150

male_WT = read_bed("Egr1CC_peak_MaleEgr1_VS_MaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed")
female_WT = read_bed("Egr1CC_peak_FemaleEgr1_VS_FemaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed")
male_vs_female = read_bed("Egr1CC_peak_MaleEgr1_VS_FemaleEgr1_MACC2_YchromFiltered_window300_bg005_p05_TTAAp001_052125.bed")
female_vs_male = read_bed("Egr1CC_peak_FemaleEgr1_VS_MaleEgr1_MACC2_YchromFiltered_window300_bg005_p05_TTAAp001_052125.bed")

print("Input peak counts:")
print(f"  Male WT Egr1 peaks: {len(male_WT)}")
print(f"  Female WT Egr1 peaks: {len(female_WT)}")
print(f"  Male-biased differential peaks: {len(male_vs_female)}")
print(f"  Female-biased differential peaks: {len(female_vs_male)}")

male_WT_slop = slop_bed(male_WT, slop_bp)
female_WT_slop = slop_bed(female_WT, slop_bp)
combined = pd.concat([male_WT_slop, female_WT_slop], ignore_index=True)
combined_merged = merge_bed(combined)
combined_merged.to_csv("combined_peaks.bed", sep="\t", index=False, header=False)
print(f"Saved combined_peaks.bed with {len(combined_merged)} merged regions (slop {slop_bp} bp).")


In [ ]:
male_vs_female_slop = slop_bed(male_vs_female, slop_bp)
female_vs_male_slop = slop_bed(female_vs_male, slop_bp)

male_overlap_df = overlapping_regions(combined_merged, male_vs_female_slop)
female_overlap_df = overlapping_regions(combined_merged, female_vs_male_slop)

male_overlap_df.to_csv("male_biased_overlaps.bed", sep="\t", index=False, header=False)
female_overlap_df.to_csv("female_biased_overlaps.bed", sep="\t", index=False, header=False)

print(f"Saved male_biased_overlaps.bed: {len(male_overlap_df)} regions")
print(f"Saved female_biased_overlaps.bed: {len(female_overlap_df)} regions")


In [ ]:
region_files = ["combined_peaks.bed", "male_biased_overlaps.bed", "female_biased_overlaps.bed"]
matrix_output = "Egr1CC_overlaps_shared_matrix_fixed.gz"
cmd = [
    '/ref/rmlab/software/conda_envs/deeptools_env/bin/computeMatrix', 'reference-point',
    "--referencePoint", "center",
    "-b", "1000",
    "-a", "1000",
    "--binSize", "10",
    "-R",
    *region_files,
    "-S",
    "../cpm_bigwigs/Male_Egr1_CPM.bw",
    "../cpm_bigwigs/Female_Egr1_CPM.bw",
    "--missingDataAsZero",
    "-p", "8",
    "-o", matrix_output
]
print("Running computeMatrix with combined and sex-biased regions...")
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")


In [ ]:
plot_output = "Egr1CC_overlaps_with_shared_heatmap_fixed.pdf"
cmd = [
    '/ref/rmlab/software/conda_envs/deeptools_env/bin/plotHeatmap',
    "-m", "Egr1CC_overlaps_shared_matrix_fixed.gz",
    "-o", plot_output,
    "--colorMap", "Blues", "RdPu",
    "--refPointLabel", "center",
    "--regionsLabel", "Shared", "Male-biased", "Female-biased",
    "--samplesLabel", "Male Egr1 CC", "Female Egr1 CC",
    "--xAxisLabel", "distance from peak center (bp)",
    "--yAxisLabel", "Peaks",
    "--zMin", "0",
    "--zMax", "5",
    "--heatmapHeight", "12",
    "--dpi", "150"
]
print("Running plotHeatmap...")
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Return code: {result.returncode}")
print(f"Heatmap saved to {plot_output}")
